In [ ]:
# Qwen3.5-0.8B Stage A2-arb-route: inference-only causal routing diagnostic (learned/shuffled/hard/easy)


In [ ]:
# Cell 0 — environment / P100 proof + mount check
import os, shutil, sys
from pathlib import Path

print('python', sys.version.split()[0], '(torch loads in Cell 1 with a P100-compatible build)')
!nvidia-smi --query-gpu=name,memory.total,memory.free,compute_cap --format=csv 2>/dev/null || nvidia-smi 2>/dev/null | head -20
print('working disk:', shutil.disk_usage('/kaggle/working'))
inp = sorted(str(p) for p in Path('/kaggle/input').glob('*')) if Path('/kaggle/input').exists() else []
print('/kaggle/input mounts:', inp if inp else 'NONE (real-PLE will abort until PLE dataset is attached)')
secret_value_0 = os.environ.get('HF_TOKEN')
secret_value_1 = os.environ.get('KAGGLE_API_TOKEN')
if not secret_value_0 or not secret_value_1:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        secret_value_0 = secret_value_0 or user_secrets.get_secret('HF_TOKEN')
        secret_value_1 = secret_value_1 or user_secrets.get_secret('KG_TOKEN')
    except Exception:
        pass
assert secret_value_0, 'set HF_TOKEN or attach the Kaggle HF_TOKEN secret'
assert secret_value_1, 'set KAGGLE_API_TOKEN or attach the Kaggle KG_TOKEN secret'
os.environ['HF_TOKEN'] = secret_value_0
os.environ['KAGGLE_API_TOKEN'] = secret_value_1


In [ ]:
# Cell 1 — deps (P100-compatible torch BEFORE first torch import)
import subprocess, sys
try:
    _cap=subprocess.run(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader'],capture_output=True,text=True,timeout=60).stdout.strip().splitlines()[0].strip()
except Exception: _cap=''
print('compute_cap:', _cap or 'unknown')
%pip install -q "transformers==5.17.0" datasets safetensors huggingface_hub matplotlib accelerate
if _cap.startswith('6.'):
    subprocess.run([sys.executable,'-m','pip','install','-q','--index-url','https://download.pytorch.org/whl/cu118','torch==2.5.1+cu118'],check=True)
    print('pinned P100 torch (cu118, sm_60 kernels)')
    subprocess.run([sys.executable,'-m','pip','install','-q','--index-url','https://download.pytorch.org/whl/cu118','torchvision==0.20.1+cu118'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','--index-url','https://download.pytorch.org/whl/cu118','torchaudio==2.5.1+cu118'],check=True)
    print('pinned matching torchvision+torchaudio (qwen3_5 modeling pulls both via media utils)')
import torch
assert torch.cuda.is_available(), 'need a GPU accelerator'
torch.zeros(1).cuda()  # fail fast if the build lacks sm_60 kernels
print('torch', torch.__version__, '| cap', torch.cuda.get_device_capability(0))
import transformers, datasets, safetensors, huggingface_hub
print(transformers.__version__, datasets.__version__, safetensors.__version__, huggingface_hub.__version__)

In [ ]:
# Cell 2 — single config block (edit here only)
from dataclasses import dataclass

@dataclass(frozen=True)
class Cfg:
    SOURCE_ID: str = 'Qwen/Qwen3.8-Flash-Next-FP8'
    TARGET_ID: str = 'Qwen/Qwen3.5-0.8B'
    MEM_DIM: int = 2560
    HIDDEN: int = 1024
    N_LAYERS: int = 24
    NGRAM: int = 3
    HEADS_PER_NGRAM: int = 8   # 16 address slots per token
    ROW_DIM: int = 160         # per-slot row dim; 16*160=2560
    ROWS_PER_PART: int = 2_500_012
    VOCAB_BASE: int = 20_000_000
    SEED: int = 1234           # PLE hash seed — never change
    EOS: int = 248044  # config <|endoftext|>: PLE-training terminator (NOT chat <|im_end|> 248046)
    VOCAB: int = 248320
    SEQ: int = 512
    VAL_FAST_TOKENS: int = 65536    # 128 x 512
    VAL_FULL_TOKENS: int = 524288   # 1024 x 512 (35B parity)
    DATASET_ID: str = 'HuggingFaceFW/fineweb-edu'  # FIRST experiment: FineWeb-Edu ONLY
    DATASET_CONFIG: str = 'sample-10BT'
    SMOKE_TOKENS: int = 5120    # correctness pass only (10 x 512); sweep stays OFF
    LR: float = 3e-5
    WD: float = 0.01
    WARMUP_FRAC: float = 0.05
    CKPTS: tuple = (100000, 250000, 500000)  # threshold-crossing (not exact multiples)
    PLACEMENTS: tuple = ((2,), (8,), (2, 8))  # zero-based IDX, 35B convention
    BRANCHES: int = 1
    GAMMA_INIT: float = 1e-3   # 0.0 = identity test mode
    EVAL_SEED: int = 1234
    EVAL_BS: int = 8
    EVAL_TARGET_S: int = 7200  # 2h soft target: A2 is cheap MC logprob, not generative
    EVAL_HARD_S: int = 32400  # 9h hard guard with persist margin (well below 12h limit)
    EVAL_MC_N: int = 1000  # per MC task before auto-reduce (min 250)
    EVAL_MC_MIN: int = 250
    EVAL_DOMAIN_TOKENS: int = 32768  # 64 x 512 per domain
    ARMS_500K: tuple = ('disabled', 'random', 'permuted', 'real')  # controlled comparison
    ARM_1M: str = 'real-1m'  # exploratory only, never in the causal set

C = Cfg()
def describe(layers): return f"IDX {list(layers)} = HUMAN {[l+1 for l in layers]}"
print(C)
print('placements:', ' / '.join(describe(l) for l in C.PLACEMENTS))
for sites, R in [(1,1),(2,1),(1,4),(2,4)]:
    n = sites*(R*C.MEM_DIM*C.HIDDEN + C.MEM_DIM*C.HIDDEN) + sites*R + sites
    print(f'sites={sites} R={R}: ~{n/1e6:.2f}M trainable')
print('controlled arms @500K:', C.ARMS_500K, '| exploratory:', C.ARM_1M)
print('A2: HellaSwag/PIQA/ARC-E/LAMBADA logprob + 5-domain NLL; cheap suite only')


In [ ]:
# Cell 3 — EXACT source addressing (port of src/qwen36_ple/hashing.py — do not modify)
import math
import torch

MASK64=(1<<64)-1; GAMMA=0x9E3779B97F4A7C15; M1=0xBF58476D1CE4E5B9; M2=0x94D049BB133111EB; LPRIME=10007

def splitmix64(v):
    v=(v+GAMMA)&MASK64; v=((v^(v>>30))*M1)&MASK64; v=((v^(v>>27))*M2)&MASK64; return (v^(v>>31))&MASK64

def layer_multipliers(vocab, ngram, ple_idx=0, seed=1234):
    mmax=((1<<63)-1)//max(vocab,1); hb=max(1,mmax//2); base=seed+LPRIME*ple_idx
    return tuple(2*(splitmix64((base+GAMMA*(i+1))&MASK64)%hb)+1 for i in range(ngram))

def is_prime(v):
    if v<2: return False
    if v%2==0: return v==2
    return all(v%d for d in range(3, math.isqrt(v)+1, 2))

def nth_prime_after(s, k):
    p=s
    for _ in range(k):
        p+=1
        while not is_prime(p): p+=1
    return p

def head_layout(ngram=3, hpn=8, base=20_000_000, ple_idx=0):
    n=(ngram-1)*hpn
    sizes=tuple(nth_prime_after(base-1, ple_idx*n+h+1) for h in range(n))
    off=[]; t=0
    for s in sizes: off.append(t); t+=s
    return sizes, tuple(off)

def shift_right_ignore_eos(ids, shift, eos):
    if shift==0: return ids
    B,L=ids.shape; pos=torch.arange(L, device=ids.device)
    eos_pos=torch.where(ids==eos, pos, -1); prev_inc=torch.cummax(eos_pos,1).values
    prev=torch.cat([eos_pos.new_full((B,1),-1), prev_inc[:,:-1]],1)
    inseg=pos.unsqueeze(0)-(prev+1); src=pos-shift
    sh=ids.gather(1, src.clamp_min(0).unsqueeze(0).expand(B,-1))
    valid=(inseg>=shift)&(src.unsqueeze(0)>=0)
    return torch.where(valid, sh, ids.new_full((), eos))

def ngram_indices(ids, eos_token_id=248044, vocab_size=248320, ngram_size=3, heads_per_ngram=8,
                    vocab_size_base=20_000_000, ple_layer_index=0, seed=1234):
    ids=ids.long()
    mult=torch.tensor(layer_multipliers(vocab_size, ngram_size, ple_layer_index, seed), device=ids.device)
    sizes, offs=head_layout(ngram_size, heads_per_ngram, vocab_size_base, ple_layer_index)
    sizes=torch.tensor(sizes, device=ids.device); offs=torch.tensor(offs, device=ids.device)
    sh=[shift_right_ignore_eos(ids,s,eos_token_id) for s in range(ngram_size)]
    blocks=[]
    for ng in range(2, ngram_size+1):
        st=(ng-2)*heads_per_ngram; mixed=sh[0]*mult[0]
        for p in range(1,ng): mixed=torch.bitwise_xor(mixed, sh[p]*mult[p])
        blocks.append(torch.remainder(mixed.unsqueeze(-1), sizes[st:st+heads_per_ngram])+offs[st:st+heads_per_ngram])
    return torch.cat(blocks,-1)  # [B,L,16] GLOBAL PLE addresses, never raw token ids

_a=ngram_indices(torch.tensor([[1,2,3,4,5]])); _b=ngram_indices(torch.tensor([[1,2,3,4,5]]))
assert torch.equal(_a,_b) and _a.shape==(1,5,16)
SIZES, OFFS = head_layout()
print('hash ok; slots/head addrs e.g.', tuple(_a[0,2,:4].tolist()), '| head0 range', (OFFS[0], OFFS[0]+SIZES[0]))
def addresses(token_cpu):
    '''ONLY path from tokens to PLE rows: exact source hashes -> global head addresses.'''
    return ngram_indices(token_cpu, eos_token_id=C.EOS, vocab_size=C.VOCAB,
                         ngram_size=C.NGRAM, heads_per_ngram=C.HEADS_PER_NGRAM,
                         vocab_size_base=C.VOCAB_BASE, ple_layer_index=0, seed=C.SEED)

In [ ]:
# Cell 4 — shared-value reader (hidden=1024; reductions stay FP32 so fp16 backbone is safe)
import math
import torch
from torch import nn

def rms_norm(x, eps=1e-6):  # always FP32 reduction, cast back: fp16-safe
    return x.float().mul(torch.rsqrt(x.float().square().mean(-1, keepdim=True)+eps)).to(x.dtype)

class SharedValueReader(nn.Module):
    def __init__(self, mem_dim=2560, hidden=1024, branches=1, gamma=0.0):
        super().__init__(); self.mem_dim=mem_dim; self.hidden=hidden; self.branches=branches
        self.keys=nn.ModuleList(nn.Linear(mem_dim, hidden, bias=False) for _ in range(branches))
        self.value=nn.Linear(mem_dim, hidden, bias=False)
        self.beta=nn.Parameter(torch.zeros(branches))
        self.gamma=nn.Parameter(torch.tensor(float(gamma)))
        self.last_gate=None
    def stats(self):
        if self.last_gate is None: return None
        g=self.last_gate.float()
        return {'mean':g.mean().item(),'std':g.std(correction=0).item(),'near_zero':(g<0.01).float().mean().item()}
    def forward(self, h, m):
        assert h.shape[:-1]==m.shape[:-1], (h.shape, m.shape)
        h_dtype=h.dtype; h=h.float(); m=m.float()  # backbone may be fp16; reader computes FP32
        q=rms_norm(h); v=self.value(m); gs=[]
        for b,proj in enumerate(self.keys):
            k=rms_norm(proj(m))
            s=(q.float()*k.float()).sum(-1)/math.sqrt(self.hidden)
            gs.append(torch.sigmoid(s+self.beta[b].float()).to(v.dtype))
        g=torch.stack(gs,0)
        o=(g.unsqueeze(-1)*v.unsqueeze(0)).mean(0)
        self.last_gate=g.detach()
        return (h+self.gamma*o).to(h_dtype)  # residual back to backbone dtype; params stay FP32

_r=SharedValueReader(gamma=0.0); _h=torch.randn(1,4,1024); _m=torch.randn(1,4,2560)
assert torch.equal(_r(_h,_m),_h)
print('reader identity ok; R=1 params:', sum(p.numel() for p in _r.parameters()))

In [ ]:
# Cell 5 — injection hooks (IDX convention) + layer helper
import torch
from torch import nn

def decoder_layers(model):
    for path in ['model.layers','language_model.layers','transformer.h']:
        o=model
        try:
            for a in path.split('.'): o=getattr(o,a)
            if len(o)==24 or len(o)>0: return o
        except Exception: pass
    raise RuntimeError('decoder layers not found')

class ReaderInjection(nn.Module):
    '''layers = zero-based IDX list, exactly like the 35B run (e.g. (2,) = third block).'''
    def __init__(self, model, layers, mem_dim=2560, hidden=1024, branches=1, gamma=0.0):
        super().__init__()
        self.idx=tuple(layers)
        self.readers=nn.ModuleDict({str(l):SharedValueReader(mem_dim,hidden,branches,gamma) for l in layers})
        self.memory=None; self.handles=[]
        dec=decoder_layers(model)
        assert len(dec)==C.N_LAYERS, f'decoder count {len(dec)} != {C.N_LAYERS} — wrong hook target'
        for l in layers:
            self.handles.append(dec[l].register_forward_pre_hook(self._hook(str(l)), with_kwargs=True))
        print(f'inject at IDX {list(layers)} = HUMAN {[l+1 for l in layers]}')
    def _hook(self,name):
        def fn(mod,args,kw):
            if self.memory is None: return args,kw
            m=self.memory
            L=args[0].shape[1] if args else kw['hidden_states'].shape[1]
            if m.shape[1]!=L: m=m[:,:L]
            if args: return (self.readers[name](args[0],m),*args[1:]),kw
            kw['hidden_states']=self.readers[name](kw['hidden_states'],m); return args,kw
        return fn
    def set_memory(self,m): self.memory=m
    def close(self):
        [h.remove() for h in self.handles]; self.handles.clear()

print('injection ok')

In [ ]:
# Cell 6 — PLE stores: real (mount-only, exact scale or abort) + calibrated controls
import json, os
from pathlib import Path
import torch
from safetensors import safe_open

PLE_TMPL='model.language_model.layers.1.ple.ple_embedding.ngram_embedding.shard_{p}.weight'
PLE_SCALE='model.language_model.layers.1.ple.ple_embedding.ngram_embedding.weight_scale'

def rss_mb():
    '''Host RSS in MiB (Linux /proc; -1 if unavailable). Proves bounded RAM.'''
    try:
        with open('/proc/self/status') as _f:
            for _line in _f:
                if _line.startswith('VmRSS:'): return float(_line.split()[1])/1024
    except Exception: pass
    try:
        import resource; return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024
    except Exception: return -1.0

def find_ple_manifests():
    hits=sorted(Path('/kaggle/input').glob('*/manifest.json'))+sorted(Path('/kaggle/input').glob('*/*/manifest.json'))
    out=[]
    for h in hits:
        try:
            m=json.loads(h.read_text())
            if isinstance(m, dict) and 'parts' in m: out.append(h)
        except Exception: pass
    return out

class MountPLE:
    '''Real frozen PLE (row-level mmap, 35B-validated). REQUIRES /kaggle/input mount. Never downloads. Never caches parts: each lookup fetches ONLY needed rows via safetensors get_slice runs, so host RAM stays bounded after all 128 parts are touched.'''
    def __init__(self, manifest=None):
        manifests=[Path(manifest)] if manifest else find_ple_manifests()
        if not manifests or not all(p.exists() for p in manifests):
            raise RuntimeError('Real-PLE ABORT: no /kaggle/input PLE dataset attached. Attach the pinned shards+manifest.json dataset(s) first; refusing to download 48.7 GiB into /kaggle/working.')
        self.part_paths={}; revs=set()
        for mp in manifests:
            m=json.loads(mp.read_text())
            if not revs: self.rpp=m.get('rows_per_part',2500012); self.rd=m.get('row_dim',160)
            revs.add(m.get('ple_revision','unknown'))
            for k,v in m['parts'].items():
                p=mp.parent/v
                if not p.exists(): continue  # each dataset holds only its own shards
                if int(k) in self.part_paths:
                    assert self.part_paths[int(k)].name==p.name, f'part {k} filename clash'
                    continue
                self.part_paths[int(k)]=p
        assert len(revs)==1, f'mixed PLE revisions: {revs}'
        self.ple_revision=revs.pop()
        assert len(self.part_paths)==128, f'need all 128 parts, have {len(self.part_paths)}'
        missing=[str(p) for p in self.part_paths.values() if not p.exists()]
        if missing: raise RuntimeError(f"Real-PLE ABORT: {len(missing)} shard files missing, e.g. {missing[0]}")
        self.scale=self._resolve_scale()  # exact scale or abort — no fallback constant
        self.calls=0; self.rows_read=0
        self.parts_touched=set()  # cumulative DISTINCT parts; no part tensors ever held
        print(f'mounted PLE rev={self.ple_revision} parts={len(self.part_paths)} scale={self.scale}')
    def _resolve_scale(self):
        for f in sorted(set(self.part_paths.values())):
            try:
                with safe_open(f, framework='pt', device='cpu') as fh:
                    if PLE_SCALE in fh.keys():
                        return float(fh.get_tensor(PLE_SCALE).float().mean())
            except Exception: pass
        raise RuntimeError('Real-PLE ABORT: weight_scale tensor not found in pinned shards/index. Refusing hardcoded fallback.')
    def stats(self):
        return {'calls': self.calls, 'rows_read': self.rows_read,
                'parts_touched': len(self.parts_touched), 'held_part_tensors': 0,
                'scale': self.scale, 'rss_MiB': round(rss_mb(), 1)}
    @staticmethod
    def tensor_name(part): return PLE_TMPL.format(p=part)
    def lookup(self, indices):  # indices = GLOBAL head addresses [..,16] from ngram_indices()
        '''Row-level mmap reads: group deduped addresses by part (sorted), fetch ONLY
        needed rows as contiguous get_slice runs, dequantize the gathered rows. Full
        part tensors (~381 MiB each) are never materialized or cached.'''
        shape=indices.shape; flat=indices.detach().cpu().long().reshape(-1)
        uniq, inv=torch.unique(flat, return_inverse=True)  # dedup repeated addresses
        parts=torch.div(uniq, self.rpp, rounding_mode='floor'); local=uniq%self.rpp
        table=torch.empty(uniq.numel(), self.rd, dtype=torch.float32)
        for p in torch.unique(parts).tolist():  # ascending part order (page-friendly)
            pos=torch.nonzero(parts==p).flatten()
            lrows=local.index_select(0, pos); srows, sidx=torch.sort(lrows)
            runs=[]; a=int(srows[0]); prev=a
            for r in srows[1:].tolist():
                if r==prev+1: prev=r
                else: runs.append((a, prev+1)); a=prev=r
            runs.append((a, prev+1))
            path=self.part_paths.get(p)
            if path is None: raise FileNotFoundError(f'PLE part {p} not in manifest')
            with safe_open(str(path), framework='pt', device='cpu') as fh:
                sl=fh.get_slice(self.tensor_name(p))
                got=torch.cat([sl[x:y].to(torch.float32) for x, y in runs])*self.scale
            table.index_copy_(0, pos.index_select(0, sidx), got)  # got aligns with srows
        self.calls+=1; self.rows_read+=uniq.numel(); self.parts_touched.update(torch.unique(parts).tolist())
        return table[inv].reshape(*shape, self.rd).flatten(-2)  # [..,16,160]->[..,2560]

def permute_addresses(addrs, seed=777):
    '''Deterministic per-head bijective address permutation (shared by builder + store).'''
    sizes, offs=head_layout()
    S=torch.tensor(sizes); O=torch.tensor(offs)
    A=[]; B=[]
    for h,(s,o) in enumerate(zip(sizes, offs)):
        a=int(splitmix64((seed+10007*(h+1))&MASK64)%(s-1))+1
        b=int(splitmix64(((seed^0x9E3779B97F4A7C15)+7919*(h+1))&MASK64)%s)
        assert a%s!=0, 'A must be coprime to prime head size'
        A.append(a); B.append(b)
    A=torch.tensor(A); B=torch.tensor(B)
    f=addrs.long().cpu()
    H=f.shape[-1]
    O=O[:H]; A=A[:H]; B=B[:H]; S=S[:H]
    return O+((f-O)*A+B)%S

class RandomPLE:
    '''Calibrated per-head deterministic control: same global address -> same 160-d row, every call.
    Means/stds are per-head scalars measured from real rows in the frozen working set.
    16 rows concatenate to 2560-d. No 0.06 fallback.'''
    def __init__(self, head_means, head_stds, seed=0, row_dim=160):
        assert len(head_means)==16 and len(head_stds)==16, 'need 16 per-head stats'
        self.means=[float(m) for m in head_means]
        self.stds=[float(s) for s in head_stds]
        assert all(s>0 for s in self.stds), 'stds must be positive (calibrated)'
        self.seed=seed; self.rd=row_dim
        _sizes,_offs=head_layout()
        self._S=list(_sizes); self._O=list(_offs)
    def _head_of(self, a):
        for h in range(16):
            if self._O[h]<=a<self._O[h]+self._S[h]: return h
        raise ValueError('address outside head ranges')
    def _rows_for(self, uniq, heads):
        rows=[]
        for a,h in zip(uniq.tolist(), heads.tolist()):
            g=torch.Generator(); g.manual_seed((self.seed*1000003+int(a))%2**63)
            rows.append(torch.randn(self.rd, generator=g)*self.stds[h]+self.means[h])
        return torch.stack(rows)
    def lookup(self, indices):
        shape=indices.shape; flat=indices.detach().cpu().long().reshape(-1)
        uniq, inv=torch.unique(flat, return_inverse=True)
        heads=torch.tensor([self._head_of(int(a)) for a in uniq.tolist()])
        table=self._rows_for(uniq, heads)
        return table[inv].reshape(*shape, self.rd).flatten(-2)

def calibrate_head_stats(compact, n_per_head=4096, seed=0):
    '''Measure per-head mean/std from representative real rows in the compact working set.
    Samples n_per_head rows per head from the frozen prefix (train + full-val).'''
    sizes, offs=head_layout()
    addrs=compact.addrs.to(torch.int64)
    g=torch.Generator().manual_seed(seed)
    means=[]; stds=[]
    for h in range(16):
        lo=offs[h]; hi=offs[h]+sizes[h]
        pos=torch.nonzero((addrs>=lo)&(addrs<hi)).flatten()
        assert len(pos)>=n_per_head, f'head {h} only {len(pos)} rows'
        pick=pos[torch.randint(0,len(pos),(n_per_head,),generator=g)]
        rows=compact.rows.index_select(0, pick).to(torch.float32)*compact.scale
        means.append(float(rows.mean()))
        stds.append(float(rows.std(correction=0)))
    print('calibrated head means:', [round(m,6) for m in means])
    print('calibrated head stds:', [round(s,6) for s in stds])
    return means, stds

class PermutedPLE:
    '''Bijective per-head permutation: off_h + ((a-off_h)*A_h + B_h) % size_h.
    Preserves head ranges (sizes are prime, A_h % size_h != 0 so gcd=1 i.e. coprime) and table distribution.
    Wraps a compact working-set cache, never /kaggle/input random reads during training.'''
    def __init__(self, base, seed=777):
        self.b=base; self.seed=seed
        sizes, offs=head_layout()
        self.S=torch.tensor(sizes); self.O=torch.tensor(offs)
        A=[]; B=[]
        for h,(s,o) in enumerate(zip(sizes, offs)):
            a=int(splitmix64((seed+10007*(h+1))&MASK64)%(s-1))+1
            b=int(splitmix64(((seed^0x9E3779B97F4A7C15)+7919*(h+1))&MASK64)%s)
            assert a%s!=0, 'A must be coprime to prime head size'
            A.append(a); B.append(b)
        self.A=torch.tensor(A); self.B=torch.tensor(B)
    def lookup(self, indices):
        H=indices.shape[-1]; f=indices.long().cpu()
        O=self.O[:H]; A=self.A[:H]; B=self.B[:H]; S=self.S[:H]
        return self.b.lookup((O+((f-O)*A+B)%S).to(indices.device) if indices.is_cuda else (O+((f-O)*A+B)%S))

class CompactPLE:
    '''Compact working set for one frozen token prefix: sorted int32 addresses + fp8 rows (mmap).
    Bit-exact vs MountPLE: same bytes, same scale, same dequant formula. No 48.7 GiB traffic.'''
    def __init__(self, directory):
        import json as _json
        d=Path(directory)
        meta=_json.loads((d/'compact.json').read_text())
        n=meta['address_count']
        self.addrs=torch.from_file(str(d/'addrs.u32'), shared=True, size=n, dtype=torch.int32)
        raw=torch.from_file(str(d/'rows.u8'), shared=True, size=n*meta['row_dim'], dtype=torch.uint8)
        self.rows=raw.view(torch.float8_e4m3fn).view(n, meta['row_dim'])
        self.scale=float(meta['scale']); self.meta=meta
        self.ple_revision=meta.get('ple_revision')
        print('compact PLE: %d rows, %.2f GiB mapped, scale=%g' % (n, (d/'rows.u8').stat().st_size/1024**3, self.scale))
    def lookup(self, indices):
        shape=indices.shape; flat=indices.detach().cpu().long().reshape(-1)
        assert bool((flat>=0).all()) and int(flat.max())<2**31
        f32=flat.to(torch.int32)
        pos=torch.searchsorted(self.addrs, f32)
        posc=pos.clamp(max=len(self.addrs)-1)
        assert bool((self.addrs[posc]==f32).all()), 'compact miss: address outside frozen prefix'
        out=self.rows.index_select(0, posc).to(torch.float32)*self.scale
        return out.reshape(*shape, self.rows.shape[-1]).flatten(-2)



print('stores ok; manifests:', [str(p) for p in find_ple_manifests()])


In [ ]:
# Cell 7 — tokenizer verification: SEMANTIC id-space check (SPEC #15)
# Pinned-rev forensics: both model.vocab = 248044 entries with 0 id diffs; source-only
# ids are 7 audio added-tokens (248070-248076) above the target max: no collision.
# Config eos = 248044 (<|endoftext|>) on BOTH. AutoTokenizer.eos_token_id may report
# chat <|im_end|> 248046 instead: a chat-template default, NOT the PLE-training
# terminator. Hashing keeps training constants (vocab 248320 / eos 248044 / seed 1234).
import json
from transformers import AutoTokenizer
from huggingface_hub import HfApi, hf_hub_download

tok=secret_value_0; assert tok, 'Attach HF_TOKEN in Settings -> Secrets'
api=HfApi(token=tok)
trev=api.model_info(C.TARGET_ID).sha; srev=api.model_info(C.SOURCE_ID).sha
print('target rev', trev[:12], '| source rev', srev[:12])
tt=AutoTokenizer.from_pretrained(C.TARGET_ID, token=tok, revision=trev)
st=AutoTokenizer.from_pretrained(C.SOURCE_ID, token=tok, revision=srev)
def raw_vocab(path):
    tj=json.load(open(path, encoding='utf-8'))
    m=dict(tj['model']['vocab'])
    for a in tj.get('added_tokens', []): m[a['content']]=a['id']
    return m
tm=raw_vocab(hf_hub_download(C.TARGET_ID,'tokenizer.json',revision=trev,token=tok))
sm=raw_vocab(hf_hub_download(C.SOURCE_ID,'tokenizer.json',revision=srev,token=tok))
diff={k for k in tm if k in sm and tm[k]!=sm[k]}
extra_t={k for k in tm if k not in sm}
smax=max(tm.values())
src_only={k: sm[k] for k in sm if k not in tm}
print('mapping diffs:', len(diff), '| target-only:', len(extra_t), '| source-only:', len(src_only))
assert not diff and not extra_t, 'target id space must match source ids exactly (native addressing)'
assert all(v>smax for v in src_only.values()), 'source-only ids must sit above target range'
assert sm.get('<|endoftext|>')==C.EOS and tm.get('<|endoftext|>')==C.EOS, 'training eos must be <|endoftext|> both sides'
probe='The quick brown fox 0123456789 function print(){} <|im_start|>x<|im_end|>'
assert tt.encode(probe, add_special_tokens=False)==st.encode(probe, add_special_tokens=False), 'probe encodings differ — STOP'
print('NATIVE token-ID addressing VALID: identical ids; training terminator eos =', C.EOS)


In [ ]:
# Cell 8 — immutable validation artifacts (built ONCE, never inside training) + FineWeb-Edu-only stream
import hashlib, json
from array import array
from pathlib import Path
from datasets import load_dataset

WORK=Path('/kaggle/working/ple-08b'); WORK.mkdir(parents=True, exist_ok=True)
VALDIR=WORK/'val-frozen-v1'; VALDIR.mkdir(exist_ok=True)

def _write_val(name, tokens_u32, meta_extra):
    tp=VALDIR/f'tokens-{name}.uint32le'; mp=VALDIR/f'validation-{name}.json'
    raw=tokens_u32.tobytes(); digest=hashlib.sha256(raw).hexdigest()
    meta={'name':name,'token_count':len(tokens_u32),'seq':512,'count':len(tokens_u32)//512,
            'tokens_sha256':digest, **meta_extra}
    if mp.exists():
        old=json.loads(mp.read_text())
        if old!=meta or tp.read_bytes()!=raw:
            raise RuntimeError(f'Immutable validation {name} differs — refusing to overwrite')
        print(f"reuse frozen val-{name} sha={digest[:16]} n={len(tokens_u32)}"); return meta
    tp.write_bytes(raw); mp.write_text(json.dumps(meta,indent=2)); print(f'wrote frozen val-{name} sha={digest[:16]}')
    return meta

def build_validation_artifacts(tok):
    '''Prefix-consistent: stream 524288 FineWeb-Edu tokens once; fast = first 65536 slice.'''
    from huggingface_hub import HfApi
    import os
    api=HfApi(token=secret_value_0)
    drev=api.dataset_info(C.DATASET_ID).sha; trev2=api.model_info(C.TARGET_ID).sha
    need_full=VALDIR/'validation-full.json'; need_fast=VALDIR/'validation-fast.json'
    if need_full.exists() and need_fast.exists():
        return json.loads(need_fast.read_text()), json.loads(need_full.read_text())
    ds=load_dataset(C.DATASET_ID, C.DATASET_CONFIG, split='train', streaming=True, revision=drev)
    arr=array('I'); docs=0
    for row in ds:
        docs+=1; t=row.get('text') or ''
        if t.strip(): arr.extend(tok.encode(t, add_special_tokens=False)); arr.append(C.EOS)  # training terminator
        if len(arr)>=C.VAL_FULL_TOKENS: del arr[C.VAL_FULL_TOKENS:]; break
    assert len(arr)==C.VAL_FULL_TOKENS, len(arr)
    base={'dataset':C.DATASET_ID,'config':C.DATASET_CONFIG,'dataset_rev':drev,'target_rev':trev2,'docs':docs,'skip_docs':docs}
    fast_arr=array('I', arr[:C.VAL_FAST_TOKENS])
    mf=_write_val('fast', fast_arr, base); mF=_write_val('full', arr, base)
    return mf, mF

def load_validation(name):
    m=json.loads((VALDIR/f'validation-{name}.json').read_text())
    raw=(VALDIR/f'tokens-{name}.uint32le').read_bytes()
    assert hashlib.sha256(raw).hexdigest()==m['tokens_sha256'], 'val checksum mismatch'
    a=array('I'); a.frombytes(raw)
    import torch
    return torch.tensor(a,dtype=torch.long).view(-1,512), m



print('build with build_validation_artifacts(tok); read with load_validation("fast"/"full")')

In [ ]:
# Cell 9 — frozen Qwen3.5-0.8B via full-config VLM-compat CausalLM (vision frozen, text path): float16 FIRST
import os, torch
from transformers import AutoConfig, AutoModelForCausalLM

tok=secret_value_0; assert tok, 'Attach HF_TOKEN in Settings -> Secrets'
from huggingface_hub import HfApi
trev=HfApi(token=tok).model_info(C.TARGET_ID).sha
cfg=AutoConfig.from_pretrained(C.TARGET_ID, token=tok, revision=trev, trust_remote_code=True)
assert getattr(cfg,'model_type',None)=='qwen3_5', getattr(cfg,'model_type',None)
tconf=cfg.text_config  # dims ONLY — never pass as config= (strips auto_map, breaks class resolution)
print('hidden',tconf.hidden_size,'layers',tconf.num_hidden_layers,'vocab',tconf.vocab_size,'arch',type(cfg).__name__)
assert tconf.hidden_size==1024 and tconf.num_hidden_layers==24
from transformers import AutoTokenizer
tokenizer=AutoTokenizer.from_pretrained(C.TARGET_ID, token=tok, revision=trev, trust_remote_code=True)

try:
    from transformers.models.qwen3_5 import Qwen3_5ForCausalLM as _Q
    print('direct qwen3_5 import ok:', _Q.__name__)
except Exception:
    import traceback; traceback.print_exc()
    raise RuntimeError('qwen3_5 modeling import failed — true cause above')
ACTIVE_DTYPE=None; model=None
for dt in [torch.float16, torch.float32]:
    try:
        m=AutoModelForCausalLM.from_pretrained(C.TARGET_ID, revision=trev, token=tok,
            trust_remote_code=True, device_map={'':0}, torch_dtype=dt, low_cpu_mem_usage=True)
        m.eval(); m.requires_grad_(False)
        assert sum(1 for p in m.parameters() if p.requires_grad)==0
        ids=tokenizer('The quick brown fox jumps over the lazy dog. '*8, return_tensors='pt').input_ids[:,:64].cuda()
        with torch.inference_mode():
            lg=m(input_ids=ids,use_cache=False).logits
        assert torch.isfinite(lg.float()).all(), 'non-finite logits'
        model=m; ACTIVE_DTYPE=dt; print(f'frozen load OK in {dt} footprint={m.get_memory_footprint()/1024**3:.2f} GiB')
        print('model class:', type(m).__name__, '| decoder blocks:', len(decoder_layers(m)))
        del ids, lg; break
    except Exception as e:
        print(f'{dt} rejected: {str(e)[:200]}')
        try: del m
        except Exception: pass
assert model is not None and ACTIVE_DTYPE is not None
print('backbone = ACTIVE_DTYPE:', ACTIVE_DTYPE)
print('reader params/compute = FP32 (AdamW FP32; norms/scores FP32)')
print('PLE dequant/output = FP32')
print('reader residual output is cast back to backbone dtype')

In [ ]:
# Cell 11b — validation prep: build immutable artifacts ONCE, then load both (idempotent)
mf, mfull = build_validation_artifacts(tokenizer)
val_fast, _ = load_validation("fast")
val_full, _ = load_validation("full")

print("validation ready")
print("fast:", mf["tokens_sha256"])
print("full:", mfull["tokens_sha256"])


In [ ]:
# Stage A2 harness: MC adapters (HellaSwag/PIQA/ARC-E/LAMBADA) + frozen domain-NLL builders.
# Logprob scoring only; no free generation in this notebook.
PRED_A2 = []
import json, re, subprocess, sys, time, zlib
import torch, torch.nn.functional as F
EVAL_SEED = 1234
EVAL_BS = 8
DOMAINS = ('general', 'code', 'math', 'scientific', 'multilingual')
def _ds_rev(ds_id):
    from huggingface_hub import HfApi
    try:
        return HfApi(token=secret_value_0).dataset_info(ds_id).sha
    except Exception:
        return None
def _load_split(ds_id, split, config=None):
    from datasets import load_dataset
    rev = _ds_rev(ds_id)
    kw = {'revision': rev} if rev else {}
    if config is None:
        return load_dataset(ds_id, split=split, **kw), rev, None
    return load_dataset(ds_id, config, split=split, **kw), rev, config
def _subset(rows, n, seed=EVAL_SEED):
    import random
    rows = list(rows)
    if len(rows) <= n:
        return rows
    rng = random.Random(seed)
    idx = list(range(len(rows)))
    rng.shuffle(idx)
    return [rows[i] for i in sorted(idx[:n])]
def _load_first(cands, split):
    errs = []
    for _cid, _cfg in cands:
        try:
            _ds, _rev, _ = _load_split(_cid, split, _cfg)
            print('dataset resolved: ' + _cid + ('/' + _cfg if _cfg else ''), flush=True)
            return _ds, _rev, _cid, _cfg
        except Exception as e:
            errs.append(_cid + ': ' + str(e)[:120])
    raise RuntimeError('no candidate resolved: ' + ' | '.join(errs))
def _hellaswag_rows(n):
    ds, rev, ds_id, cfg = _load_first([('Rowan/hellaswag', None), ('hellaswag', None)], 'validation')
    rows = []
    for r in ds:
        ctx = r.get('ctx') or ((r.get('ctx_a') or '') + ' ' + (r.get('ctx_b') or '')).strip()
        ends = list(r.get('endings') or [])
        try:
            lab = int(str(r.get('label')).strip())
        except Exception:
            continue
        if not ctx or len(ends) != 4:
            continue
        rows.append({'ctx': ctx, 'endings': ends, 'label': lab})
    rows = _subset(rows, n)
    return rows, {'dataset': ds_id, 'config': cfg, 'rev': str(rev), 'n': len(rows), 'scoring': 'logprob-sum+length-norm'}
def _piqa_rows(n):
    # Config-first: ybisk/piqa offers a plain_text subset (legacy script path is
    # rejected by datasets>=4 when no config is given). Direct-jsonl fallback last.
    try:
        ds, rev, ds_id, cfg = _load_first([('ybisk/piqa', 'plain_text'), ('piqa', 'plain_text'), ('ybisk/piqa', None), ('piqa', None)], 'validation')
        ds_id_out, rev_out = ds_id, str(rev)
    except Exception:
        import urllib.request
        from datasets import load_dataset as _ld
        base = 'https://yonatanbisk.com/piqa/data/'
        ds = _ld('json', data_files={'validation': base + 'dev.jsonl'}, split='validation')
        with urllib.request.urlopen(base + 'dev-labels.lst', timeout=120) as _r:
            _labs = _r.read().decode('utf-8').split()
        ds_id_out, rev_out, cfg = 'yonatanbisk.com/piqa/data', 'direct-jsonl', None
        print('dataset resolved: yonatanbisk.com/piqa/data (direct jsonl)', flush=True)
        rows = []
        for r, _lab in zip(ds, _labs):
            g = (r.get('goal') or '').strip()
            s = [(r.get('sol1') or '').strip(), (r.get('sol2') or '').strip()]
            try:
                lab = int(str(_lab).strip())
            except Exception:
                continue
            if not g or not s[0] or not s[1] or lab not in (0, 1):
                continue
            rows.append({'goal': g, 'sols': s, 'label': lab})
        rows = _subset(rows, n)
        return rows, {'dataset': ds_id_out, 'config': cfg, 'rev': rev_out, 'n': len(rows), 'scoring': 'logprob-sum+length-norm'}
    rows = []
    for r in ds:
        g = (r.get('goal') or r.get('question') or r.get('query') or '').strip()
        s1 = (r.get('sol1') or '').strip()
        s2 = (r.get('sol2') or '').strip()
        if (not s1 or not s2) and isinstance(r.get('choices'), list) and len(r.get('choices')) >= 2:
            s1, s2 = str(r['choices'][0]).strip(), str(r['choices'][1]).strip()
        try:
            lab = int(str(r.get('label', r.get('answer', ''))).strip())
        except Exception:
            continue
        if not g or not s1 or not s2 or lab not in (0, 1):
            continue
        rows.append({'goal': g, 'sols': [s1, s2], 'label': lab})
    rows = _subset(rows, n)
    return rows, {'dataset': ds_id, 'config': cfg, 'rev': str(rev), 'n': len(rows), 'scoring': 'logprob-sum+length-norm'}
def _arc_rows(n):
    ds, rev, ds_id, cfg = _load_first([('allenai/ai2_arc', 'ARC-Easy'), ('ai2_arc', 'ARC-Easy')], 'validation')
    rows = []
    for r in ds:
        q = (r.get('question') or '').strip()
        ch = r.get('choices')
        try:
            if isinstance(ch, dict) and 'text' in ch and 'label' in ch:
                texts = [str(t).strip() for t in ch['text']]
                labs = [str(v).strip() for v in ch['label']]
            elif isinstance(ch, list):
                texts = [str(c.get('text', '')).strip() for c in ch]
                labs = [str(c.get('label', '')) for c in ch]
            else:
                continue
            key = str(r.get('answerKey')).strip()
            li = labs.index(key)
        except Exception:
            continue
        if not q or len(texts) < 2:
            continue
        rows.append({'question': q, 'options': texts, 'label': li, 'answerKey': key})
    rows = _subset(rows, n)
    return rows, {'dataset': ds_id, 'config': cfg, 'rev': str(rev), 'n': len(rows), 'scoring': 'logprob-sum+length-norm'}
def _lambada_rows(n):
    ds, rev, ds_id, cfg = _load_first([('EleutherAI/lambada_openai', None), ('lambada', None)], 'test')
    rows = []
    for r in ds:
        t = (r.get('text') or '').strip()
        parts = t.split()
        if len(parts) < 10:
            continue
        rows.append({'context': ' '.join(parts[:-1]), 'target': parts[-1], 'text': t})
    rows = _subset(rows, n)
    return rows, {'dataset': ds_id, 'config': cfg, 'rev': str(rev), 'n': len(rows), 'scoring': 'teacher-forced last-word NLL+acc'}
def _domain_cands(domain):
    if domain == 'general':
        return [('wikitext', 'wikitext-103-raw-v1', 'test', 'text'), ('wikitext', 'wikitext-2-raw-v1', 'test', 'text')]
    if domain == 'code':
        return [('edward-io/starcoderdata-repo', None, 'train', 'code'), ('codeparrot/github-code-clean', None, 'train', 'code'), ('bigcode/the-stack-smol', None, 'train', 'code')]
    if domain == 'math':
        return [('openai/gsm8k', 'main', 'test', 'qa'), ('gsm8k', 'main', 'test', 'qa')]
    if domain == 'scientific':
        return [('allenai/sciq', None, 'test', 'sci'), ('sciq', None, 'test', 'sci')]
    if domain == 'multilingual':
        return [('wikipedia', '20220301.de', 'train', 'text'), ('HuggingFaceFW/fineweb-2', 'deu_Latn', 'train', 'text')]
    raise ValueError('unknown domain ' + domain)
def _extract_domain_text(domain, row):
    if domain in ('general', 'multilingual'):
        return (row.get('text') or '')
    if domain == 'code':
        return (row.get('content') or row.get('code') or row.get('text') or '')
    if domain == 'math':
        return ((row.get('question') or '') + '\n' + (row.get('answer') or ''))
    if domain == 'scientific':
        return ((row.get('support') or '') + '\n' + (row.get('question') or '') + '\n' + (row.get('correct_answer') or row.get('answer') or ''))
    return ''
def _write_domain(name, tokens_u32, meta_extra):
    from array import array as _array
    import hashlib as _hl
    ddir = WORK / 'domain-frozen-v1'
    ddir.mkdir(parents=True, exist_ok=True)
    tp = ddir / f'tokens-{name}.uint32le'
    mp = ddir / f'domain-{name}.json'
    raw = tokens_u32.tobytes()
    digest = _hl.sha256(raw).hexdigest()
    meta = {'name': name, 'token_count': len(tokens_u32), 'seq': 512, 'count': len(tokens_u32) // 512, 'tokens_sha256': digest, **meta_extra}
    if mp.exists():
        old = json.loads(mp.read_text())
        if old != meta or tp.read_bytes() != raw:
            raise RuntimeError(f'Immutable domain {name} differs — refusing to overwrite')
        print(f"reuse frozen domain-{name} sha={digest[:16]} n={len(tokens_u32)}")
        return meta
    tp.write_bytes(raw)
    mp.write_text(json.dumps(meta, indent=2))
    print(f'wrote frozen domain-{name} sha={digest[:16]}')
    return meta
def build_domain_blocks(tok, domain, n_tokens):
    from array import array as _array
    from datasets import load_dataset as _ld
    from huggingface_hub import HfApi as _Api
    assert domain in DOMAINS, domain
    ddir = WORK / 'domain-frozen-v1'
    ddir.mkdir(parents=True, exist_ok=True)
    mp = ddir / f'domain-{domain}.json'
    if mp.exists():
        return load_domain(domain)
    errs = []
    for ds_id, cfg, split, kind in _domain_cands(domain):
        try:
            api = _Api(token=secret_value_0)
            drev = api.dataset_info(ds_id).sha
            kw = {'revision': drev} if drev else {}
            if cfg is None:
                ds = _ld(ds_id, split=split, streaming=True, **kw)
            else:
                ds = _ld(ds_id, cfg, split=split, streaming=True, **kw)
            arr = _array('I')
            docs = 0
            for row in ds:
                docs += 1
                t = _extract_domain_text(domain, row)
                if t and t.strip():
                    arr.extend(tok.encode(t, add_special_tokens=False))
                    arr.append(C.EOS)
                if len(arr) >= n_tokens:
                    del arr[n_tokens:]
                    break
            assert len(arr) == n_tokens, (domain, len(arr))
            meta = _write_domain(domain, arr, {'dataset': ds_id, 'config': cfg, 'split': split, 'kind': kind, 'dataset_rev': drev, 'docs': docs})
            return load_domain(domain)
        except Exception as e:
            errs.append(f'{ds_id}: {str(e)[:140]}')
    raise RuntimeError('no domain candidate resolved for ' + domain + ': ' + ' | '.join(errs))
def load_domain(domain):
    import hashlib as _hl
    from array import array as _array
    ddir = WORK / 'domain-frozen-v1'
    m = json.loads((ddir / f'domain-{domain}.json').read_text())
    raw = (ddir / f'tokens-{domain}.uint32le').read_bytes()
    assert _hl.sha256(raw).hexdigest() == m['tokens_sha256'], 'domain checksum mismatch ' + domain
    a = _array('I')
    a.frombytes(raw)
    import torch as _torch
    return _torch.tensor(a, dtype=_torch.long).view(-1, 512), m
print('A2 harness ready (4 MC adapters + 5 frozen domain builders, logprob only)')


In [ ]:
# Stage A2-arb-route: inference-only causal routing diagnostic on frozen REAL-5M R=1 (IDX 2+8).
# No training, no weight changes, no R4. Uses deployed-B gate outputs only.
# Question: is routing the same memory budget to uncertain positions causally better than to easy ones?
# Arms (IDX2 fixed 1.25 everywhere; only IDX8 placement changes, per-sequence multiset exact):
#   learned  = B deployment as-is (must reproduce arb-B numbers: invariant below)
#   shuffled = per-sequence pinned permutation of alpha8 (same multiset, location destroyed)
#   hard     = largest alpha8 to highest disabled-entropy scored positions (leftovers to unscored)
#   easy     = largest alpha8 to lowest disabled-entropy scored positions (leftovers to unscored)
# Unscored positions (block tail, prompt prefix) always receive the smallest leftovers in order.
# Entropy comes from the disabled pure-backbone pass of the same sequence.
# Eval: full-val NLL (1024 blocks), hellaswag acc (1000), lambada acc/NLL (1000), 5 frozen domains (64 each).
# Persists: routing.json, hellaswag-<arm>.jsonl, lambada-<arm>.jsonl, config-route.json, timings-route.json.
import hashlib, json, math, time
import torch, torch.nn.functional as F
from pathlib import Path
from torch import nn
T0 = time.perf_counter()
EOUT = Path('/kaggle/working/eval-stageA2-arb-route'); EOUT.mkdir(parents=True, exist_ok=True)
EFI_MOUNT = Path('/kaggle/input/qwen-ple-reader-checkpoints')
ARB_MOUNT = Path('/kaggle/input/qwen-ple-a2-arb')
RDIR = Path('/kaggle/input/qwen-ple-a2-v34')
DLDIR = Path('/kaggle/working/ckpt-dl-arb-route'); DLDIR.mkdir(parents=True, exist_ok=True)
assert RDIR.exists(), 'results dataset missing: attach ninnix/qwen-ple-a2-v34'
assert ARB_MOUNT.exists(), 'arb dataset missing: attach ninnix/qwen-ple-a2-arb'
REF = {
    'target_rev': '2fc06364715b967f1860aea9cf38778875588b17',
    'source_rev_prefix': '236dfdf28582',
    'ple_revision': '236dfdf285828023ca3bcd3f37366c58a3469b13',
    'ckpt_sha': {'real-5m': '078d7b47b979dbdae1442cbcc15bc466f72b8942159ede8c02fad0daedb63594'},
    'ckpt_tokens': {'real-5m': 5000192},
    'arb_sha': {
        'arbitration-b.pt': '595d195a84c65acd2aa414d61726e080b54801b4676b3d7a2256ca8a7bcd9948',
        'arbitration-c.pt': '2cbdd7913413c87ed45500d4496540fcbab30e3a97d6070a54e5f38d94eef3d9',
        'arbitration-b.json': '0331a2e7b281a7469448c87edee67260cd8462412b486c6e56b9ea8b75f85230',
        'arbitration-c.json': '89bf8edeb7d2e26dfb87cef15015615e8d364222971b49fbda074bdcd020035a',
        'alpha8-stats.json': '0793e0b6899e6d27b7c37397c0374c33383477a631a83117760e031d8be9183a',
    },
    'mc': {
        'hellaswag': {'dataset': 'Rowan/hellaswag', 'rev': '218ec52e09a7e7462a5400043bb9a69a41d06b76', 'n': 1000},
        'lambada': {'dataset': 'EleutherAI/lambada_openai', 'rev': '900124bf3b8235c6daf21033af9948b3f07346c4', 'n': 1000},
    },
    'val_full_sha256': '26ffe65b1f6457d2557dbacc98197d025932057e3c183e3cf1de8eefb7c2fe7c',
    'domains': {
        'general': 'a9e12294eba002b5a4e9fce610ff39f8aed6d802e36e534e82ed367f96bb9aa5',
        'code': '735f933ca4bc694562dbb2a6ab9c227c802268f7715448eba50e172ae450c421',
        'math': 'c27f42f5f69de1aaf26020c7e14c2c141f8fa8490c5d95952c06e1c5bffa6648',
        'scientific': 'c33632cf96170ca7ae92d4013702c14097fd79bb069c76e40fb3a531091936d8',
        'multilingual': '65a26da5419aaa32520219d9d1c5909769a7a19a88ff16e85df7d69c38028a55',
    },
}
B_REF = {'val_nll': 2.865490686403562, 'hs_acc': 0.404, 'lam_acc': 0.453, 'lam_nll': 2.253693464072421,
         'general': 3.148797033350995, 'code': 1.4981278458686724, 'math': 1.4137548299218343,
         'scientific': 2.256194297581503, 'multilingual': 3.661386721288155}
ARMS = ('learned', 'shuffled', 'hard', 'easy')
assert trev == REF['target_rev'], 'tokenizer revision drift: %s' % trev
assert srev.startswith(REF['source_rev_prefix']), 'source revision drift'
print('frozen revisions ok (tokenizer/model/ple)', flush=True)
print('route: 4 arms, per-sequence budget exact, inference only; no R4, no weight updates', flush=True)
timings = {}
def _mark(name):
    timings[name] = round(time.perf_counter() - T0, 1)
    print('[t=%ds] %s' % (timings[name], name), flush=True)
def _guard():
    assert (time.perf_counter() - T0) <= C.EVAL_HARD_S - 1800, 'HARD-GUARD trip'
def _file_sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()
for _f, _h in REF['arb_sha'].items():
    assert (ARB_MOUNT / _f).exists(), 'missing arb file: ' + _f
    assert _file_sha(ARB_MOUNT / _f) == _h, 'arb bytes mismatch: ' + _f
print('arb outputs pinned (B weights + stats SHAs match)', flush=True)
def _fetch_ckpt():
    need = ['protocol.json', 'real-5m-r1.reader.safetensors', 'real-5m-r1.run.json']
    if EFI_MOUNT.exists() and all((EFI_MOUNT / f).exists() for f in need):
        print('checkpoints: mounted dataset (schema-validated)', flush=True)
        return EFI_MOUNT
    import os, subprocess, sys
    _env = dict(os.environ); _env['KAGGLE_API_TOKEN'] = secret_value_1
    for _f in need:
        _r = subprocess.run([sys.executable, '-m', 'kaggle', 'datasets', 'download', '-d', 'ninnix/qwen-ple-reader-checkpoints', '-f', _f, '-p', str(DLDIR), '--force'], capture_output=True, text=True, env=_env, timeout=1200)
        assert _r.returncode == 0 and (DLDIR / _f).exists(), 'checkpoint download failed: ' + _f
    print('checkpoints: api download (mount missing)', flush=True)
    return DLDIR
CKDIR = _fetch_ckpt()
_proto = json.loads((CKDIR / 'protocol.json').read_text())
assert _proto.get('ple_revision') == REF['ple_revision'], 'PLE revision drift'
def _load_sfc(pt_name, run_name, expect_tokens, expect_sha):
    from safetensors.torch import load_file
    run = json.loads((CKDIR / run_name).read_text())
    assert run.get('token_count', run.get('tokens')) == expect_tokens, 'budget mismatch: ' + run_name
    h = _file_sha(CKDIR / pt_name)
    assert h == expect_sha, 'bytes mismatch: ' + pt_name
    return load_file(str(CKDIR / pt_name), device='cpu'), h
st5, _h5 = _load_sfc('real-5m-r1.reader.safetensors', 'real-5m-r1.run.json', REF['ckpt_tokens']['real-5m'], REF['ckpt_sha']['real-5m'])
print('frozen REAL-5M R=1 weights ready', flush=True)
_mark('checkpoints ready')
V = {}
for _f in ['hellaswag.jsonl', 'lambada.jsonl']:
    assert (RDIR / _f).exists(), 'missing frozen input: ' + _f
V['rows'] = {}
for _t in ['hellaswag', 'lambada']:
    V['rows'][_t] = [json.loads(x) for x in (RDIR / f'{_t}.jsonl').read_text().splitlines() if x.strip()]
print('frozen v34 MC rows loaded', flush=True)
hs_rows, hs_meta = _hellaswag_rows(C.EVAL_MC_N)
lam_rows, lam_meta = _lambada_rows(C.EVAL_MC_N)
for _name, _rows, _meta in [('hellaswag', hs_rows, hs_meta), ('lambada', lam_rows, lam_meta)]:
    _ref = REF['mc'][_name]
    assert _meta['dataset'] == _ref['dataset'] and str(_meta['rev']) == _ref['rev'], 'dataset drift: ' + _name
    assert len(_rows) == _ref['n'] == len(V['rows'][_name]), 'row count drift: ' + _name
print('MC manifests identical to v34 (hellaswag+lambada)', flush=True)
val_full_chk, mfull_chk = load_validation('full')
assert mfull_chk['tokens_sha256'] == REF['val_full_sha256'], 'frozen full-val drift'
dom_blocks = {}
for _d in ('general', 'code', 'math', 'scientific', 'multilingual'):
    _b, _m = build_domain_blocks(tokenizer, _d, C.EVAL_DOMAIN_TOKENS)
    assert _m['tokens_sha256'] == REF['domains'][_d], 'domain token drift: ' + _d
    assert _b.shape[0] == 64, 'domain block drift: ' + _d
    dom_blocks[_d] = _b
print('frozen full-val (1024 blocks) + 5 domains ok', flush=True)
_mark('frozen inputs verified')
tokenizer.padding_side = 'left'
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = C.EOS
inj = ReaderInjection(model, [2, 8], C.MEM_DIM, C.HIDDEN, 1, C.GAMMA_INIT).to('cuda')
inj.load_state_dict({k: v.to('cuda') for k, v in st5.items()})
inj.requires_grad_(False)
model.requires_grad_(False)
assert sum(1 for p in inj.parameters() if p.requires_grad) == 0, 'reader must stay grad-free'
assert sum(1 for p in model.parameters() if p.requires_grad) == 0, 'backbone must stay frozen'
orig_g2 = float(inj.readers['2'].gamma.detach().cpu())
orig_g8 = float(inj.readers['8'].gamma.detach().cpu())
assert abs(orig_g2 - 0.06166713312268257) < 1e-6 and abs(orig_g8 - 0.07537073642015457) < 1e-6, '5M weights mismatch'
print('frozen reader ok: IDX2=%.6f IDX8=%.6f' % (orig_g2, orig_g8), flush=True)
class GateOnly(nn.Module):
    def __init__(self, w, b):
        super().__init__()
        self.w = w.float()
        self.b = float(b)
    def alpha8(self, h8):
        hn = rms_norm(h8.detach().float())
        return 0.5 * torch.sigmoid((hn * self.w.to(hn.device)).sum(-1) + self.b)
stB = torch.load(str(ARB_MOUNT / 'arbitration-b.pt'), map_location='cpu', weights_only=True)
assert tuple(stB['w'].shape) == (1024,), 'arb w shape drift'
gB = GateOnly(stB['w'], stB['b'])
print('deployed-B gate loaded (frozen, inference only)', flush=True)
SHUF = torch.Generator().manual_seed(1234)
class RouteHooks(nn.Module):
    # Deployed-B reproduction with swappable IDX8 placement. IDX2 fixed 1.25 always.
    # mode off = pure pass-through (disabled baseline). Otherwise w8 is rearranged alpha8.
    # scored_mask marks gate positions that predict a scored target; leftovers go to the rest.
    def __init__(self, model, frozen, gate):
        super().__init__()
        self.gate = gate
        self.fr2 = frozen.readers['2']
        self.fr8 = frozen.readers['8']
        self.mode = 'off'
        self.memory = None
        self.ent = None
        self.scored = None
        self.applied = {}
        dec = decoder_layers(model)
        self.handles = [
            dec[2].register_forward_pre_hook(self._hook2(), with_kwargs=True),
            dec[8].register_forward_pre_hook(self._hook8(), with_kwargs=True),
        ]
    def _place(self, a8):
        T = a8.shape[0]
        if self.mode == 'learned':
            return a8
        if self.mode == 'shuffled':
            return a8[torch.randperm(T, generator=SHUF)]
        desc = torch.argsort(a8, descending=True)
        sc = torch.nonzero(self.scored).flatten()
        un = torch.nonzero(~self.scored).flatten()
        order = torch.argsort(self.ent[sc], descending=(self.mode == 'hard'))
        w = torch.empty_like(a8)
        w[sc[order]] = a8[desc[:len(sc)]]
        w[un] = a8[desc[len(sc):]]
        return w
    def _hook2(self):
        def fn(mod, args, kw):
            if self.mode == 'off' or self.memory is None:
                return args, kw
            h = args[0] if args else kw['hidden_states']
            m = self.memory if self.memory.shape[1] == h.shape[1] else self.memory[:, :h.shape[1]]
            aug = self.fr2(h, m)
            out = h + 1.25 * (aug - h)
            if args:
                return (out, *args[1:]), kw
            kw['hidden_states'] = out
            return args, kw
        return fn
    def _hook8(self):
        def fn(mod, args, kw):
            if self.mode == 'off' or self.memory is None:
                return args, kw
            h = args[0] if args else kw['hidden_states']
            m = self.memory if self.memory.shape[1] == h.shape[1] else self.memory[:, :h.shape[1]]
            a8 = self.gate.alpha8(h).reshape(-1)
            w = self._place(a8)
            _s, _n = self.applied.get(self.mode, (0.0, 0))
            self.applied[self.mode] = (_s + float(w.sum()), _n + int(w.numel()))
            aug = self.fr8(h, m)
            out = h + w.to(h.dtype).unsqueeze(-1) * (aug - h)
            if args:
                return (out, *args[1:]), kw
            kw['hidden_states'] = out
            return args, kw
        return fn
    def set_memory(self, m):
        self.memory = m
    def close(self):
        [h.remove() for h in self.handles]; self.handles.clear()
def _build_compact_route(pdir, seqs):
    import hashlib as _hl
    from array import array as _array
    from safetensors import safe_open as _so
    _real = []
    with torch.inference_mode():
        for _s in seqs:
            _real.append(ngram_indices(_s.unsqueeze(0).cpu()).reshape(-1))
    uniq = torch.unique(torch.cat(_real)).long().cpu()
    print('route union addresses: %d' % len(uniq), flush=True)
    del _real
    _uh = _hl.sha256(_array('I', uniq.tolist()).tobytes()).hexdigest()
    _ms = MountPLE()
    assert _ms.ple_revision == REF['ple_revision'], 'PLE revision drift'
    _uq = uniq.sort().values
    pdir.mkdir(parents=True, exist_ok=True)
    _af = (pdir / 'addrs.u32').open('wb'); _rf = (pdir / 'rows.u8').open('wb')
    _ha = _hl.sha256(); _hr = _hl.sha256(); _n = 0; _prev = -1
    _pa = torch.div(_uq, C.ROWS_PER_PART, rounding_mode='floor'); _lo = _uq % C.ROWS_PER_PART
    _bf = {}
    for _p in torch.unique(_pa).tolist():
        _bf.setdefault(str(_ms.part_paths[_p]), []).append(_p)
    for _fp in sorted(_bf):
        with _so(_fp, framework='pt', device='cpu') as _fh:
            for _p in sorted(_bf[_fp]):
                _full = _fh.get_slice(MountPLE.tensor_name(_p))[:]
                _pos = torch.nonzero(_pa == _p).flatten()
                _au = _uq.index_select(0, _pos)
                assert int(_au[0]) > _prev; _prev = int(_au[-1])
                _ab = _array('I', _au.tolist()).tobytes()
                _rb = bytes(_full.index_select(0, _lo.index_select(0, _pos)).view(torch.uint8).flatten().tolist())
                _af.write(_ab); _rf.write(_rb); _ha.update(_ab); _hr.update(_rb); _n += _pos.numel()
    _af.close(); _rf.close()
    (pdir / 'compact.json').write_text(json.dumps({'format': 'qwen-ple-compact-a2-arb-route', 'version': 1, 'address_count': _n, 'uniq_sha256': _uh, 'row_dim': C.ROW_DIM, 'scale': float(_ms.scale), 'ple_revision': _ms.ple_revision, 'addrs_sha256': _ha.hexdigest(), 'rows_sha256': _hr.hexdigest()}, indent=2))
    _cc = CompactPLE(pdir)
    _g = torch.Generator().manual_seed(0)
    _samp = _uq[torch.randint(0, len(_uq), (2048,), generator=_g)].reshape(128, 16)
    assert (_cc.lookup(_samp) - _ms.lookup(_samp)).abs().max().item() == 0.0
    return _cc
_seq_all = [val_full_chk[i] for i in range(val_full_chk.shape[0])]
for _d in dom_blocks:
    for _bi in range(dom_blocks[_d].shape[0]):
        _seq_all.append(dom_blocks[_d][_bi])
for r in hs_rows:
    p = tokenizer(r['ctx'], return_tensors='pt', add_special_tokens=False)['input_ids'][0]
    for e in r['endings']:
        _seq_all.append(torch.cat([p, tokenizer((' ' + e.strip() if not e.startswith(' ') else e), return_tensors='pt', add_special_tokens=False)['input_ids'][0]])[:512])
for r in lam_rows:
    _seq_all.append(torch.cat([tokenizer(r['context'], return_tensors='pt', add_special_tokens=False)['input_ids'][0], tokenizer((' ' + r['target'].strip() if not r['target'].startswith(' ') else r['target']), return_tensors='pt', add_special_tokens=False)['input_ids'][0]])[:512])
_cgr = _build_compact_route(WORK / 'compact-a2-arb-route', _seq_all)
del _seq_all
_mark('compact ready (route union)')
rt = RouteHooks(model, inj, gB)
CHUNK = 128
def _disabled_forward(ids):
    rt.mode = 'off'
    rt.memory = None
    rt.ent = None
    rt.scored = None
    inj.set_memory(None)
    with torch.inference_mode():
        return model(input_ids=ids.to('cuda'), use_cache=False).logits
def _routed_forward(ids, mode, ent, scored):
    rt.mode = mode
    rt.ent = ent
    rt.scored = scored
    with torch.inference_mode():
        rt.set_memory(_cgr.lookup(addresses(ids.cpu())).to('cuda'))
        return model(input_ids=ids.to('cuda'), use_cache=False).logits
def _full_entropy(lg_cpu, n_tgt):
    ents = []
    for _s in range(0, n_tgt, CHUNK):
        _e = min(_s + CHUNK, n_tgt)
        lp = torch.log_softmax(lg_cpu[_s:_e], dim=-1)
        pr = lp.exp()
        ents.append((-pr * lp).sum(-1))
        del lp, pr
    return torch.cat(ents)
_dp = hs_rows[0]
_d1a = _disabled_forward(torch.cat([tokenizer(_dp['ctx'], return_tensors='pt', add_special_tokens=False)['input_ids'][0], tokenizer(' ' + _dp['endings'][0], return_tensors='pt', add_special_tokens=False)['input_ids'][0]])[:512].unsqueeze(0))
_d1b = _disabled_forward(torch.cat([tokenizer(_dp['ctx'], return_tensors='pt', add_special_tokens=False)['input_ids'][0], tokenizer(' ' + _dp['endings'][0], return_tensors='pt', add_special_tokens=False)['input_ids'][0]])[:512].unsqueeze(0))
assert torch.equal(_d1a.cpu(), _d1b.cpu()), 'eval path not deterministic'
print('eval determinism ok', flush=True)
del _d1a, _d1b, _dp
_mark('scoring ready')
import random as _r
def _boot_diff(vals_a, vals_b, n_boot=10000, seed=1234):
    dd = [a - b for a, b in zip(vals_a, vals_b)]
    rng = _r.Random(seed)
    n = len(dd)
    reps = sorted(sum(dd[rng.randrange(n)] for _ in range(n)) / n for _ in range(n_boot))
    ge = sum(1 for x in reps if x >= 0.0)
    le = sum(1 for x in reps if x <= 0.0)
    return {'mean': sum(dd) / n, 'lo': reps[int(0.025 * n_boot)], 'hi': reps[int(0.975 * n_boot) - 1], 'p': min(1.0, 2.0 * min(ge, le) / n_boot)}
def _block_nll_all(blocks):
    sums = {a: 0.0 for a in ARMS}
    ns = {a: 0 for a in ARMS}
    for _bi in range(blocks.shape[0]):
        _guard()
        b = blocks[_bi]
        L = int(b.shape[0])
        lg_d = _disabled_forward(b.unsqueeze(0)).float().cpu()[0]
        ent = _full_entropy(lg_d, L - 1)
        scored = torch.ones(L, dtype=torch.bool)
        scored[-1] = False
        for _arm in ARMS:
            lg = _routed_forward(b.unsqueeze(0), _arm, ent, scored).float()
            s = F.cross_entropy(lg[:, :-1].reshape(-1, lg.shape[-1]), b[1:].reshape(-1).to(lg.device), reduction='sum').item()
            sums[_arm] += s
            ns[_arm] += L - 1
            del lg
        del lg_d, ent
        if (_bi + 1) % 256 == 0:
            print('blocks %d/%d' % (_bi + 1, blocks.shape[0]), flush=True)
    return {_a: sums[_a] / max(1, ns[_a]) for _a in ARMS}
def _hs_all():
    base = V['rows']['hellaswag']
    outs = {a: [] for a in ARMS}
    accs = {a: [] for a in ARMS}
    for _j, r in enumerate(hs_rows):
        _guard()
        assert r['label'] == base[_j]['label'], 'prompt drift: hellaswag idx %d' % _j
        p = tokenizer(r['ctx'], return_tensors='pt', add_special_tokens=False)['input_ids'][0]
        opts = []
        for e in r['endings']:
            t = tokenizer(e if e.startswith(' ') else ' ' + e, return_tensors='pt', add_special_tokens=False)['input_ids'][0]
            opts.append(torch.cat([p, t], dim=0))
        lps = {a: [] for a in ARMS}
        for _o, _ids in enumerate(opts):
            L = int(_ids.shape[0])
            lg_d = _disabled_forward(_ids.unsqueeze(0)).float().cpu()[0]
            ent = _full_entropy(lg_d, L - 1)
            t = tokenizer(r['endings'][_o] if r['endings'][_o].startswith(' ') else ' ' + r['endings'][_o], return_tensors='pt', add_special_tokens=False)['input_ids'][0]
            scored = torch.zeros(L, dtype=torch.bool)
            scored[max(0, L - 1 - int(t.shape[0])):L - 1] = True
            del lg_d
            for _arm in ARMS:
                lg = _routed_forward(_ids.unsqueeze(0), _arm, ent, scored).float()[0]
                lp = torch.log_softmax(lg[p.shape[0] - 1:L - 1], dim=-1)
                lps[_arm].append(lp.gather(1, _ids[p.shape[0]:L].to(lg.device).unsqueeze(1)).sum().item())
                del lg, lp
            del ent
        for _arm in ARMS:
            pred = max(range(len(opts)), key=lambda i: lps[_arm][i])
            rec = dict(base[_j])
            rec[_arm] = {'logprobs': lps[_arm], 'pred': pred, 'correct': int(pred == r['label'])}
            outs[_arm].append(rec)
            accs[_arm].append(int(pred == r['label']))
        if (_j + 1) % 250 == 0:
            print('hs %d/1000' % (_j + 1), flush=True)
    return outs, {_a: sum(accs[_a]) / max(1, len(accs[_a])) for _a in ARMS}, accs
def _lam_all():
    base = V['rows']['lambada']
    outs = {a: [] for a in ARMS}
    accs = {a: [] for a in ARMS}
    nlls = {a: [] for a in ARMS}
    for _j, r in enumerate(lam_rows):
        _guard()
        assert r['target'] == base[_j]['target'], 'prompt drift: lambada idx %d' % _j
        p = tokenizer(r['context'], return_tensors='pt', add_special_tokens=False)['input_ids'][0]
        t = tokenizer(r['target'] if r['target'].startswith(' ') else ' ' + r['target'], return_tensors='pt', add_special_tokens=False)['input_ids'][0]
        _ids = torch.cat([p, t], dim=0)
        L = int(_ids.shape[0])
        lg_d = _disabled_forward(_ids.unsqueeze(0)).float().cpu()[0]
        ent = _full_entropy(lg_d, L - 1)
        scored = torch.zeros(L, dtype=torch.bool)
        scored[max(0, L - 1 - int(t.shape[0])):L - 1] = True
        del lg_d
        for _arm in ARMS:
            lg = _routed_forward(_ids.unsqueeze(0), _arm, ent, scored).float()[0]
            lp = torch.log_softmax(lg[p.shape[0] - 1:L - 1], dim=-1)
            tt = _ids[p.shape[0]:L].to(lg.device)
            nll = -lp.gather(1, tt.unsqueeze(1)).sum().item()
            acc = int(bool((lp.argmax(-1).to('cpu') == tt.to('cpu')).all()))
            rec = dict(base[_j])
            rec[_arm] = {'nll': nll, 'nll_per_tok': nll / max(1, int(tt.shape[0])), 'correct': acc, 'ntok': int(tt.shape[0])}
            outs[_arm].append(rec)
            accs[_arm].append(acc)
            nlls[_arm].append(nll / max(1, int(tt.shape[0])))
            del lg, lp
        del ent
        if (_j + 1) % 250 == 0:
            print('lambada %d/1000' % (_j + 1), flush=True)
    return outs, {_a: sum(accs[_a]) / max(1, len(accs[_a])) for _a in ARMS}, {_a: sum(nlls[_a]) / max(1, len(nlls[_a])) for _a in ARMS}, accs, nlls
val_nll = _block_nll_all(val_full_chk)
print('full-val NLL: %s' % ({k: round(v, 5) for k, v in val_nll.items()}), flush=True)
_mark('full-val routed (1024 blocks x 5 passes)')
dom_nll = {}
for _d in ('general', 'code', 'math', 'scientific', 'multilingual'):
    _guard()
    dom_nll[_d] = _block_nll_all(dom_blocks[_d])
    print('domain %s NLL: %s' % (_d, {k: round(v, 5) for k, v in dom_nll[_d].items()}), flush=True)
_mark('domains routed (5 x 64 blocks x 5 passes)')
hs_outs, hs_acc, hs_corr = _hs_all()
print('HS acc: %s' % ({k: round(v, 4) for k, v in hs_acc.items()}), flush=True)
_mark('hellaswag routed (1000 x 4 opts x 5 passes)')
lam_outs, lam_acc, lam_nll, lam_corr, lam_n = _lam_all()
print('LAMBADA acc: %s nll: %s' % ({k: round(v, 4) for k, v in lam_acc.items()}, {k: round(v, 5) for k, v in lam_nll.items()}), flush=True)
_mark('lambada routed (1000 x 5 passes)')
for _arm in ARMS:
    (EOUT / f'hellaswag-{_arm}.jsonl').write_text('\n'.join(json.dumps(x) for x in hs_outs[_arm]))
    (EOUT / f'lambada-{_arm}.jsonl').write_text('\n'.join(json.dumps(x) for x in lam_outs[_arm]))
budget = {m: s / max(1, n) for m, (s, n) in rt.applied.items()}
_bmeans = [budget[m] for m in ARMS]
assert max(_bmeans) - min(_bmeans) < 1e-6 * max(1e-9, max(_bmeans)), 'per-arm alpha8 budget diverged'
print('budget: per-arm mean applied a8=%s (multisets exact)' % ({k: round(v, 6) for k, v in budget.items()}), flush=True)
inv = {
    'learned_val_nll_vs_B': float(val_nll['learned'] - B_REF['val_nll']),
    'learned_hs_acc_vs_B': float(hs_acc['learned'] - B_REF['hs_acc']),
    'learned_lam_acc_vs_B': float(lam_acc['learned'] - B_REF['lam_acc']),
    'learned_lam_nll_vs_B': float(lam_nll['learned'] - B_REF['lam_nll']),
}
assert abs(inv['learned_val_nll_vs_B']) < 5e-3, 'learned arm does not reproduce arb-B val NLL'
assert abs(inv['learned_hs_acc_vs_B']) <= 0.003, 'learned arm does not reproduce arb-B HS acc'
assert abs(inv['learned_lam_nll_vs_B']) < 5e-3, 'learned arm does not reproduce arb-B LAMBADA NLL'
print('INVARIANT ok: learned reproduces arb-B (dval=%.2e dhs=%.4f dlam-nll=%.2e)' % (inv['learned_val_nll_vs_B'], inv['learned_hs_acc_vs_B'], inv['learned_lam_nll_vs_B']), flush=True)
boot = {}
for _arm in ('shuffled', 'hard', 'easy'):
    boot[f'hs-{_arm}-vs-learned'] = _boot_diff(hs_corr[_arm], hs_corr['learned'])
    boot[f'lam-acc-{_arm}-vs-learned'] = _boot_diff(lam_corr[_arm], lam_corr['learned'])
    boot[f'lam-nll-{_arm}-vs-learned'] = _boot_diff(lam_n[_arm], lam_n['learned'])
    print('HS %s-vs-learned: dacc %+.4f p=%.4g | LAM-nll: dnll %+.5f p=%.4g' % (_arm, boot[f'hs-{_arm}-vs-learned']['mean'], boot[f'hs-{_arm}-vs-learned']['p'], boot[f'lam-nll-{_arm}-vs-learned']['mean'], boot[f'lam-nll-{_arm}-vs-learned']['p']), flush=True)
(EOUT / 'bootstrap.json').write_text(json.dumps(boot, indent=2))
(EOUT / 'routing.json').write_text(json.dumps({'arms': list(ARMS), 'val_nll': val_nll, 'domains': dom_nll, 'hs_acc': hs_acc, 'lam_acc': lam_acc, 'lam_nll': lam_nll, 'budget': budget, 'invariant_vs_arbB': inv, 'note': 'per-sequence alpha8 multiset exact in all arms; only placement differs; IDX2 fixed 1.25'}, indent=2))
(EOUT / 'config-route.json').write_text(json.dumps({'seeds': {'eval': C.EVAL_SEED, 'ple': C.SEED, 'shuffle': 1234}, 'arms': list(ARMS), 'frozen_ref': REF, 'arb_B_ref': B_REF, 'note': 'inference-only routing diagnostic; backbone/PLE/reader/arb frozen; no R4'}, indent=2))
(EOUT / 'timings-route.json').write_text(json.dumps(timings, indent=2))
rt.close()
inj.close()
_mark('STAGE A2-ARB-ROUTE COMPLETE: routing is causal probe; no training; no R4')
